# 2. Do different architectures decompose an image the same way?

NAVE is architecture-agnostic: the same procedure explains CNNs and ViTs, so their explanations can
be compared directly (paper, Table 6.c). Here: ViT-Ti/16, ResNet18, ResNet50 and ViT-S/16, all
trained on ImageNet, with their default layers and the same number of concepts.

In [ ]:
# %pip install nave-xai
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cbook
from matplotlib.image import imread
from nave_xai import NAVEImage

image = imread(cbook.get_sample_data("grace_hopper.jpg", asfileobj=False))   # ships with matplotlib

def show(ax, explainer, exp, title=""):
    """An explanation map in the explainer's concept colours."""
    ax.imshow(explainer.palette[exp.explanation]); ax.set_title(title, fontsize=10); ax.axis("off")

TRAINING = "imagenet"     # "random" to skip downloads
K = 8

In [ ]:
encoders = {"ViT-Ti/16": "vitt16", "ResNet18": "resnet18", "ResNet50": "resnet50", "ViT-S/16": "vits16"}
explanations = {}
fig, axes = plt.subplots(1, 4, figsize=(16, 4.8))
for ax, (name, model) in zip(axes, encoders.items()):
    ex = NAVEImage(model=model, training=TRAINING, k=K)
    explanations[name] = ex.explain(image)
    show(ax, ex, explanations[name], name)

### How similar are the decompositions?

Concept ids are arbitrary, so explanations are compared with the *normalised mutual information*
(NMI) of their explanation maps: 1 means the same partition of the image, 0 unrelated ones.

In [ ]:
def nmi(a, b):
    a, b = a.ravel(), b.ravel()
    joint = np.zeros((a.max() + 1, b.max() + 1))
    np.add.at(joint, (a, b), 1)
    p = joint / joint.sum()
    pa, pb = p.sum(1), p.sum(0)
    nz = p > 0
    mi = (p[nz] * np.log(p[nz] / np.outer(pa, pb)[nz])).sum()
    h = lambda q: -(q[q > 0] * np.log(q[q > 0])).sum()
    return mi / np.sqrt(h(pa) * h(pb))

names = list(explanations)
M = np.array([[nmi(explanations[i].explanation, explanations[j].explanation) for j in names] for i in names])
fig, ax = plt.subplots(figsize=(5, 4.2))
im = ax.imshow(M, vmin=0, vmax=1, cmap="viridis")
ax.set_xticks(range(len(names)), names, rotation=30, ha="right"); ax.set_yticks(range(len(names)), names)
for i in range(len(names)):
    for j in range(len(names)):
        ax.text(j, i, f"{M[i, j]:.2f}", ha="center", va="center", color="w")
plt.colorbar(im); ax.set_title("NMI between explanation maps");

### Your own architectures

```python
import torchvision
convnext = torchvision.models.convnext_tiny(weights="DEFAULT")
explanations["ConvNeXt-T"] = NAVEImage(convnext, layers=["features.5", "features.7"], k=K).explain(image)
```

## Citation

If you use nave-xai, please cite the paper:

```bibtex
@inproceedings{boubekki2026nave,
  author    = {Boubekki, Ahc{\`e}ne and Fadel, Samuel G. and Mair, Sebastian},
  editor    = {Ceci, Michelangelo and Kocev, Dragi and Monreale, Anna and Vens, Celine and Flach, Peter
               and Manco, Giuseppe and Spiliopoulou, Myra and Masciari, Elio},
  title     = {Multi-depth Concept Extraction for Post-Hoc Vision Encoder Explanation},
  booktitle = {Machine Learning and Knowledge Discovery in Databases. Research Track},
  publisher = {Springer Nature Switzerland},
  address   = {Cham},
  pages     = {449--467},
  year      = {2026},
  doi       = {10.1007/978-3-032-37664-0_26},
  isbn      = {978-3-032-37664-0}
}
```

We thank Ulf Brefeld (Leuphana University Lüneburg) for lending us the computing resources without which this work would not have been possible.